In [7]:
import duckdb

In [8]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [16]:
df = con.execute(""" 
                     SELECT 
                            *
                      FROM( 
                            SELECT 
                                *
                                , ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
                            FROM bronze_z0019
                            WHERE data_ingestao >= '2026-03-03'
                 
                         )
                        where row = 1
                 
                 """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-03 20:13:27.697959,1
1,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-03-03 20:05:41.436583,1
2,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-03-03 20:05:41.436583,1
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-03 20:13:27.697959,1
4,10003,PREGO,BT50,100,60,z0019_2.csv,2026-03-03 20:13:27.697959,1


In [22]:
df_final = df.drop(columns=['nome_arquivo','data_ingestao','row'])
df_final = df_final.rename(columns={"NATBR":"ID"})
df_final = df_final.rename(columns={"MAKTX":"NM_PRODUTO"})
df_final = df_final.rename(columns={"WERKS":"ID_CATEGORIA"})
df_final = df_final.rename(columns={"MAINS":"ID_FORNECEDOR"})
df_final = df_final.rename(columns={"LABST":"VL_PRECO"})
df_final.head(10)

,ID,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10005,MACHADO,BT50,100,100
1,10001,PARAFUSO,BT10,100,100
2,10002,MARTELO,BT50,100,1500
3,10004,SERRA,BT50,100,200
4,10003,PREGO,BT50,100,60


In [23]:
df_final.dtypes

ID               str
NM_PRODUTO       str
ID_CATEGORIA     str
ID_FORNECEDOR    str
VL_PRECO         str
dtype: object

In [ ]:
df2 = df_final
df2.head(10)


,ID,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10005,MACHADO,BT50,100,100
1,10001,PARAFUSO,BT10,100,100
2,10002,MARTELO,BT50,100,1500
3,10004,SERRA,BT50,100,200
4,10003,PREGO,BT50,100,60


In [49]:
df2 = df2.astype(
    {
        'ID': int,
        'NM_PRODUTO': str,
        'ID_CATEGORIA':str,
        'ID_FORNECEDOR':int,
        'VL_PRECO':float
    }
)
df2.head(10)

,ID,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10005,MACHADO,BT50,100,100.0
1,10001,PARAFUSO,BT10,100,100.0
2,10002,MARTELO,BT50,100,1500.0
3,10004,SERRA,BT50,100,200.0
4,10003,PREGO,BT50,100,60.0


In [50]:
df2.dtypes

ID                 int64
NM_PRODUTO           str
ID_CATEGORIA         str
ID_FORNECEDOR      int64
VL_PRECO         float64
dtype: object

In [51]:
con.execute("""
    CREATE TABLE IF NOT EXISTS produtos(
            id BIGINT,
            nm_produto TEXT,    
            id_categoria TEXT,    
            id_fornecedor BIGINT,    
            vl_preco FLOAT    )
""")

In [52]:
display(df2)

,ID,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10005,MACHADO,BT50,100,100.0
1,10001,PARAFUSO,BT10,100,100.0
2,10002,MARTELO,BT50,100,1500.0
3,10004,SERRA,BT50,100,200.0
4,10003,PREGO,BT50,100,60.0


In [60]:
con.execute("insert into produtos select * from df2 ")

In [ ]:
df_resultado = con.execute(" select * from produtos").fetchdf()
df_resultado.head(10)


In [62]:
con.close()